In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import re
import random

In [ ]:
scenarios_path = Path().resolve() / "../../scenarios_inputs" / "nie" 
annotated_output_path = Path().resolve() / "../../annotated_outputs" / "nie"

print(f"scenarios_path: {scenarios_path}")
print(f"annotated_output_path: {annotated_output_path}")

In [ ]:
# Hardcoded outcome indices
temp_df = pd.read_csv("nie_primary_outcomes", encoding="utf-8")
primary_outcome_idx = dict(zip(temp_df["id"], temp_df["outcome"]))

with open(scenarios_path / "nie_scenarios.json", "r", encoding="utf-8") as f:
    master_data = json.load(f)

# Convert list to dict keyed by ID for fast lookup
master_by_id = {entry["id"]: entry for entry in master_data if entry["id"] in primary_outcome_idx}

# Regex to extract ID from filenames
pattern = re.compile(r"nie_scenarios_(\d+)_choice_\d+\.json")

# Storing extracted results
results = []

for file in annotated_output_path.glob("*.json"):
    match = pattern.match(file.name)
    if not match:
        continue  # Skip unrelated files

    scenario_id = int(match.group(1))  # Turn matching ID into integer

    # Look up scenario info from master JSON
    if scenario_id not in master_by_id:
        continue

    scenario_info = master_by_id[scenario_id]

    # Extract required fields
    scenario_text = scenario_info["text"]
    action = scenario_info["options"].get("1", None)

    # Get outcome from hardcoded list
    chosen_outcome = primary_outcome_idx.get(scenario_id, None)

    results.append({
        "id": scenario_id,
        "scenario_text": scenario_text,
        "action_choice": action,
        "outcome": chosen_outcome
    })

# Save results
df = pd.DataFrame(results)

pairs = {}
for r in results:
    pairs.setdefault(r["id"] // 2, []).append(r)

bucket_A, bucket_B = [], []

for items in pairs.values():
    random.shuffle(items)  # Randomize
    if len(items) >= 1:
        bucket_A.append(items[0])
    if len(items) >= 2:
        bucket_B.append(items[1])

pd.DataFrame(bucket_A).to_csv("nie_survey_set_A.csv", index=False)
pd.DataFrame(bucket_B).to_csv("nie_survey_set_B.csv", index=False)